In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load
train = pd.read_csv('../data/train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv('../data/store.csv', low_memory=False)
df = train.merge(store, on='Store', how='left')

# Clean
df = df[df['Open'] == 1].copy()
df = df[df['Sales'] > 0].copy()
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print("Ready.")

Shape: (844338, 18)
Ready.


In [2]:
def add_calendar_features(df):
    df = df.copy()
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfMonth'] = df['Date'].dt.day
    df['Quarter'] = df['Date'].dt.quarter
    df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)
    df['IsMonthStart'] = (df['DayOfMonth'] <= 3).astype(int)
    df['IsMonthEnd'] = (df['DayOfMonth'] >= 28).astype(int)
    df['IsDecember'] = (df['Month'] == 12).astype(int)
    return df

df = add_calendar_features(df)
print(f"New columns: {['Year','Month','Week','DayOfMonth','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsDecember']}")
print(df[['Date','Year','Month','Week','DayOfMonth','IsWeekend']].head(3))

New columns: ['Year', 'Month', 'Week', 'DayOfMonth', 'Quarter', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'IsDecember']
        Date  Year  Month  Week  DayOfMonth  IsWeekend
0 2013-01-02  2013      1     1           2          0
1 2013-01-03  2013      1     1           3          0
2 2013-01-04  2013      1     1           4          0


In [3]:
def add_lag_features(df, lags=[1, 2, 3, 7, 14, 21, 28]):
    df = df.copy()
    df = df.sort_values(['Store', 'Date']).reset_index(drop=True)
    
    for lag in lags:
        df[f'Sales_lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)
    
    return df

df = add_lag_features(df)

lag_cols = [f'Sales_lag_{l}' for l in [1, 2, 3, 7, 14, 21, 28]]
print(f"Lag columns added: {lag_cols}")
print(f"\nSample (Store 1, first rows with valid lags):")
store1_sample = df[df['Store'] == 1][['Date', 'Sales'] + lag_cols].dropna().head(3)
print(store1_sample.to_string())

Lag columns added: ['Sales_lag_1', 'Sales_lag_2', 'Sales_lag_3', 'Sales_lag_7', 'Sales_lag_14', 'Sales_lag_21', 'Sales_lag_28']

Sample (Store 1, first rows with valid lags):
         Date  Sales  Sales_lag_1  Sales_lag_2  Sales_lag_3  Sales_lag_7  Sales_lag_14  Sales_lag_21  Sales_lag_28
28 2013-02-04   7032       5970.0       5633.0       4709.0       5598.0        4127.0        4892.0        5530.0
29 2013-02-05   6049       7032.0       5970.0       5633.0       4055.0        5182.0        4881.0        4327.0
30 2013-02-06   6140       6049.0       7032.0       5970.0       3725.0        5394.0        4952.0        4486.0


In [4]:
def add_rolling_features(df, windows=[7, 14, 28]):
    df = df.copy()
    df = df.sort_values(['Store', 'Date']).reset_index(drop=True)
    
    for window in windows:
        df[f'Sales_rolling_mean_{window}'] = (
            df.groupby('Store')['Sales']
            .shift(1)
            .rolling(window)
            .mean()
            .reset_index(level=0, drop=True)
        )
        df[f'Sales_rolling_std_{window}'] = (
            df.groupby('Store')['Sales']
            .shift(1)
            .rolling(window)
            .std()
            .reset_index(level=0, drop=True)
        )
    
    return df

df = add_rolling_features(df)

roll_cols = [c for c in df.columns if 'rolling' in c]
print(f"Rolling columns added: {roll_cols}")
print(f"\nSample (Store 1):")
store1_sample = df[df['Store'] == 1][['Date', 'Sales'] + roll_cols].dropna().head(3)
print(store1_sample[['Date', 'Sales', 'Sales_rolling_mean_7', 'Sales_rolling_mean_28']].to_string())

Rolling columns added: ['Sales_rolling_mean_7', 'Sales_rolling_std_7', 'Sales_rolling_mean_14', 'Sales_rolling_std_14', 'Sales_rolling_mean_28', 'Sales_rolling_std_28']

Sample (Store 1):
         Date  Sales  Sales_rolling_mean_7  Sales_rolling_mean_28
28 2013-02-04   7032           4898.714286            5001.214286
29 2013-02-05   6049           5103.571429            5054.857143
30 2013-02-06   6140           5388.428571            5116.357143


In [5]:
def add_holiday_features(df):
    df = df.copy()
    
    # Clean StateHoliday
    df['StateHoliday'] = df['StateHoliday'].replace({'0': 0, 0: 0, 'a': 1, 'b': 1, 'c': 1})
    df['StateHoliday'] = df['StateHoliday'].astype(int)
    
    # Days before/after state holiday
    df['IsStateHoliday'] = df['StateHoliday']
    df['IsSchoolHoliday'] = df['SchoolHoliday'].astype(int)
    
    # Before/after holiday flags per store
    df = df.sort_values(['Store', 'Date']).reset_index(drop=True)
    holiday = df.groupby('Store')['IsStateHoliday']
    df['DayBeforeHoliday'] = holiday.shift(-1).fillna(0).astype(int)
    df['DayAfterHoliday'] = holiday.shift(1).fillna(0).astype(int)
    
    return df

df = add_holiday_features(df)
print("Holiday columns added: IsStateHoliday, IsSchoolHoliday, DayBeforeHoliday, DayAfterHoliday")
print(f"\nStateHoliday days in dataset: {df['IsStateHoliday'].sum()}")
print(f"SchoolHoliday days: {df['IsSchoolHoliday'].sum()}")
print(f"Days before holiday: {df['DayBeforeHoliday'].sum()}")

Holiday columns added: IsStateHoliday, IsSchoolHoliday, DayBeforeHoliday, DayAfterHoliday

StateHoliday days in dataset: 910
SchoolHoliday days: 163445
Days before holiday: 893


In [6]:
def add_competition_features(df):
    df = df.copy()
    
    # Fill missing CompetitionDistance with median
    df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
    
    # Months since competition opened
    df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(df['Year'])
    df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(df['Month'])
    
    df['CompetitionOpenMonths'] = (
        (df['Year'] - df['CompetitionOpenSinceYear']) * 12 +
        (df['Month'] - df['CompetitionOpenSinceMonth'])
    ).clip(lower=0)
    
    # Promo2 active flag
    df['Promo2Active'] = 0
    promo2_mask = df['Promo2'] == 1
    df.loc[promo2_mask, 'Promo2Active'] = 1
    
    return df

df = add_competition_features(df)
print("Competition columns added: CompetitionOpenMonths, Promo2Active")
print(f"\nCompetitionOpenMonths stats:")
print(df['CompetitionOpenMonths'].describe().round(1))

Competition columns added: CompetitionOpenMonths, Promo2Active

CompetitionOpenMonths stats:
count    844338.0
mean         42.0
std          65.2
min           0.0
25%           0.0
50%          16.0
75%          73.0
max        1386.0
Name: CompetitionOpenMonths, dtype: float64


In [7]:
# Final feature list
feature_cols = [
    'Store', 'Date', 'Sales',
    'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'StoreType', 'Assortment', 'CompetitionDistance',
    'Year', 'Month', 'Week', 'DayOfMonth', 'Quarter',
    'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'IsDecember',
    'Sales_lag_1', 'Sales_lag_2', 'Sales_lag_3',
    'Sales_lag_7', 'Sales_lag_14', 'Sales_lag_21', 'Sales_lag_28',
    'Sales_rolling_mean_7', 'Sales_rolling_std_7',
    'Sales_rolling_mean_14', 'Sales_rolling_std_14',
    'Sales_rolling_mean_28', 'Sales_rolling_std_28',
    'IsStateHoliday', 'IsSchoolHoliday',
    'DayBeforeHoliday', 'DayAfterHoliday',
    'CompetitionOpenMonths', 'Promo2Active'
]

df_features = df[feature_cols].copy()

# Save
df_features.to_parquet('../data/features.parquet', index=False)

print(f"Saved to data/features.parquet")
print(f"Shape: {df_features.shape}")
print(f"Total features: {len(feature_cols) - 3} (excluding Store, Date, Sales)")
print(f"\nNull counts:")
print(df_features.isnull().sum()[df_features.isnull().sum() > 0])

Saved to data/features.parquet
Shape: (844338, 38)
Total features: 35 (excluding Store, Date, Sales)

Null counts:
Sales_lag_1               1115
Sales_lag_2               2230
Sales_lag_3               3345
Sales_lag_7               7805
Sales_lag_14             15610
Sales_lag_21             23415
Sales_lag_28             31220
Sales_rolling_mean_7      7805
Sales_rolling_std_7       7805
Sales_rolling_mean_14    15610
Sales_rolling_std_14     15610
Sales_rolling_mean_28    31220
Sales_rolling_std_28     31220
dtype: int64
